# Hidden Insights & Executive Presentation

**Medallion Stage:** Presentation (Derived from Gold)

This notebook surfaces **10 non-obvious patterns** in IBM's HR Attrition dataset —
insights that would be missed by a standard dashboard or summary statistics.

Each insight is backed by statistical evidence and translated into an actionable
business recommendation.

---
| # | Insight | Risk Level |
|---|---|---|
| 1 | The Overtime Trap | Critical |
| 2 | The Salary Compression Cliff | High |
| 3 | The Promotion Stagnation Effect | High |
| 4 | The 1-Year Honeymoon / 3-Year Crisis | High |
| 5 | The Satisfaction Trinity | Critical |
| 6 | The Manager Loyalty Anchor | Medium |
| 7 | The Stock Option Paradox | Medium |
| 8 | The Distance Gradient | Medium |
| 9 | The Single Employee Vulnerability | Medium |
| 10 | The Business Travel Burnout | High |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

# Presentation-grade plot style
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FAFAFA',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})

PALETTE = {'stayed': '#27ae60', 'left': '#e74c3c', 'neutral': '#3498db', 'accent': '#8e44ad'}

df = pd.read_parquet(ROOT / 'data' / 'silver' / 'employee_attrition_silver.parquet')
print(f'Dataset: {len(df):,} employees  |  Attrition rate: {df["Attrition"].mean():.1%}')

---
## Insight 1 — The Overtime Trap

**Hypothesis:** Employees working overtime leave at a dramatically higher rate,
and this effect is *multiplicative* when combined with low job satisfaction.

> "Working overtime is the single strongest behavioral predictor of attrition."

In [ ]:
ot_rate  = df.groupby('OverTime')['Attrition'].mean()
rint_ot  = df[df['OverTime']==1]['Attrition'].mean()
rint_not = df[df['OverTime']==0]['Attrition'].mean()

# Overtime x Satisfaction interaction
interaction = df.groupby(['OverTime', 'JobSatisfaction'])['Attrition'].mean().unstack('OverTime')
interaction.columns = ['No OT', 'OT']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Simple bar
bars = axes[0].bar(['No Overtime', 'Overtime'], ot_rate.values * 100,
                   color=[PALETTE['stayed'], PALETTE['left']], edgecolor='white', width=0.5)
for bar, val in zip(bars, ot_rate.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=13)
axes[0].set_title('Attrition Rate: Overtime vs. No Overtime', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].set_ylim(0, 40)
rr = rint_ot / rint_not
axes[0].text(0.5, 0.92, f'Relative Risk: {rr:.1f}x', transform=axes[0].transAxes,
             ha='center', fontsize=12, color=PALETTE['left'], fontweight='bold')

# Right: Interaction heatmap
sns.heatmap(interaction * 100, annot=True, fmt='.1f', cmap='RdYlGn_r',
            ax=axes[1], linewidths=0.5, cbar_kws={'label': 'Attrition %'})
axes[1].set_title('Attrition % by Job Satisfaction × Overtime', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Overtime Status')
axes[1].set_ylabel('Job Satisfaction Score')

plt.suptitle('INSIGHT 1 — The Overtime Trap', fontsize=14, fontweight='bold', color=PALETTE['left'])
plt.tight_layout()
plt.show()

print(f'No Overtime attrition: {rint_not:.1%}')
print(f'Overtime attrition:    {rint_ot:.1%}')
print(f'Relative risk:         {rr:.1f}x')
print(f'\nMost dangerous combination: OT=Yes + JobSatisfaction=1')
print(f'  -> Attrition rate: {df[(df["OverTime"]==1) & (df["JobSatisfaction"]==1)]["Attrition"].mean():.1%}')

---
## Insight 2 — The Salary Compression Cliff

**Hypothesis:** Employees earning below their job-level median leave at twice the rate
of those earning above it — even small salary gaps compound over time.

> "It's not absolute income that drives attrition — it's relative income within peer group."

In [ ]:
df['IncomeBand'] = pd.cut(df['IncomeVsLevelMedian'], bins=[-2, -0.20, -0.05, 0.05, 0.20, 2],
                           labels=['Far Below', 'Below', 'At Median', 'Above', 'Far Above'])

comp_rates = df.groupby('IncomeBand', observed=True)['Attrition'].agg(['mean', 'count']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors = [PALETTE['left'] if r > df['Attrition'].mean() else PALETTE['stayed']
          for r in comp_rates['mean']]
bars = axes[0].bar(comp_rates['IncomeBand'], comp_rates['mean'] * 100,
                   color=colors, edgecolor='white')
axes[0].axhline(df['Attrition'].mean() * 100, color='gray', linestyle='--', label='Company avg')
for bar, row in zip(bars, comp_rates.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{row.mean:.1%}\n(n={row.count})', ha='center', va='bottom', fontsize=8.5)
axes[0].set_title('Attrition Rate by Income vs. Level Median', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=15)

# Per job level
level_comp = df.groupby(['JobLevel', 'IncomeBand'], observed=True)['Attrition'].mean().unstack('IncomeBand')
level_comp.columns = [str(c) for c in level_comp.columns]
sns.heatmap(level_comp * 100, annot=True, fmt='.0f', cmap='YlOrRd',
            ax=axes[1], linewidths=0.5, cbar_kws={'label': 'Attrition %'})
axes[1].set_title('Attrition % by Job Level × Income Band', fontweight='bold')
axes[1].set_xlabel('Income Band')
axes[1].set_ylabel('Job Level')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('INSIGHT 2 — The Salary Compression Cliff', fontsize=14, fontweight='bold', color=PALETTE['left'])
plt.tight_layout()
plt.show()

---
## Insight 3 — The Promotion Stagnation Effect

**Hypothesis:** Employees who haven't been promoted in 4+ years have significantly
higher attrition — yet this signal is invisible in aggregate satisfaction scores.

> "Employees don't leave companies. They leave stagnation."

In [ ]:
promo = df.groupby('YearsSinceLastPromotion')['Attrition'].agg(['mean', 'count']).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.fill_between(promo['YearsSinceLastPromotion'], promo['mean'] * 100,
                 alpha=0.35, color=PALETTE['left'])
ax1.plot(promo['YearsSinceLastPromotion'], promo['mean'] * 100,
         color=PALETTE['left'], linewidth=2.5, marker='o', markersize=6)
ax1.axvline(4, color='purple', linestyle='--', linewidth=1.5, label='4-year stagnation line')
ax1.set_ylabel('Attrition Rate (%)', color=PALETTE['left'], fontsize=12)
ax1.set_xlabel('Years Since Last Promotion')

ax2.bar(promo['YearsSinceLastPromotion'], promo['count'], alpha=0.25, color='gray', label='Headcount')
ax2.set_ylabel('Employee Count', color='gray')

ax1.set_title('INSIGHT 3 — Attrition Rate vs. Years Since Last Promotion\n'
              'The stagnation cliff at year 4+', fontweight='bold', fontsize=13)
ax1.legend(loc='upper left')
plt.tight_layout()
plt.show()

stag_4   = df[df['YearsSinceLastPromotion'] >= 4]['Attrition'].mean()
recent   = df[df['YearsSinceLastPromotion'] < 2]['Attrition'].mean()
print(f'Attrition — promoted within 2 years: {recent:.1%}')
print(f'Attrition — no promotion in 4+ years: {stag_4:.1%}')
print(f'Relative risk: {stag_4/recent:.1f}x')

---
## Insight 4 — The 1-Year Honeymoon / 3-Year Crisis

**Hypothesis:** Attrition peaks in years 1–3 (onboarding failure) then normalises
before spiking again at years 5–8 (career plateau). These are two *different* problems.

In [ ]:
tenure = df.groupby('YearsAtCompany')['Attrition'].agg(['mean', 'count']).reset_index()
tenure_filt = tenure[tenure['count'] >= 10]  # filter low-N years

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(tenure_filt['YearsAtCompany'], tenure_filt['mean'] * 100,
        color=PALETTE['left'], linewidth=2.5, marker='o', zorder=3)
ax.fill_between(tenure_filt['YearsAtCompany'], tenure_filt['mean'] * 100, alpha=0.25, color=PALETTE['left'])
ax.axhline(df['Attrition'].mean() * 100, color='gray', linestyle='--', alpha=0.7, label='Company avg')

# Annotate crisis zones
ax.axvspan(0.5, 3.5, alpha=0.1, color='red', label='Onboarding crisis (yr 1–3)')
ax.axvspan(4.5, 8.5, alpha=0.1, color='orange', label='Plateau crisis (yr 5–8)')

ax.set_title('INSIGHT 4 — The Dual-Crisis Tenure Curve\n'
             'Two distinct departure waves require two different retention strategies',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Years at Company')
ax.set_ylabel('Attrition Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

early   = df[df['YearsAtCompany'].between(1, 3)]['Attrition'].mean()
plateau = df[df['YearsAtCompany'].between(5, 8)]['Attrition'].mean()
stable  = df[df['YearsAtCompany'] > 10]['Attrition'].mean()
print(f'Attrition — years 1-3 (onboarding):  {early:.1%}')
print(f'Attrition — years 5-8 (plateau):     {plateau:.1%}')
print(f'Attrition — years 10+ (stable):      {stable:.1%}')

---
## Insight 5 — The Satisfaction Trinity

**Hypothesis:** When *all three* of Environment, Job, and Relationship satisfaction
are simultaneously low, attrition is near-certain. No single dimension alone predicts this.

In [ ]:
df['SatBucket'] = df[['EnvironmentSatisfaction', 'JobSatisfaction', 'RelationshipSatisfaction']].mean(axis=1)
df['AllLow'] = ((df['EnvironmentSatisfaction'] <= 2) &
                (df['JobSatisfaction']          <= 2) &
                (df['RelationshipSatisfaction'] <= 2)).astype(int)

all_low_rate  = df[df['AllLow'] == 1]['Attrition'].mean()
all_high_rate = df[(df['EnvironmentSatisfaction'] >= 3) &
                   (df['JobSatisfaction']          >= 3) &
                   (df['RelationshipSatisfaction'] >= 3)]['Attrition'].mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bucket scatter
buckets = df.groupby(pd.cut(df['SatBucket'], bins=5))['Attrition'].mean()
bucket_labels = [str(b) for b in buckets.index]
axes[0].bar(range(len(buckets)), buckets.values * 100,
            color=[PALETTE['left'] if v > 0.2 else PALETTE['stayed'] for v in buckets.values])
axes[0].set_xticks(range(len(buckets)))
axes[0].set_xticklabels(bucket_labels, rotation=20, fontsize=9)
axes[0].set_title('Attrition Rate by Satisfaction Bucket', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')

# Bar: All-Low vs. All-High vs. Mixed
groups = ['All 3 Low', 'Mixed', 'All 3 High']
rates  = [all_low_rate,
          df[(df['AllLow']==0) & ~((df['EnvironmentSatisfaction']>=3) &
             (df['JobSatisfaction']>=3) & (df['RelationshipSatisfaction']>=3))]['Attrition'].mean(),
          all_high_rate]
colors = [PALETTE['left'], PALETTE['neutral'], PALETTE['stayed']]
bars = axes[1].bar(groups, [r * 100 for r in rates], color=colors, edgecolor='white')
for bar, val in zip(bars, rates):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[1].set_title('Satisfaction Trinity Effect', fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')

plt.suptitle('INSIGHT 5 — The Satisfaction Trinity', fontsize=14, fontweight='bold', color=PALETTE['left'])
plt.tight_layout()
plt.show()

print(f'All-3-low attrition:  {all_low_rate:.1%}')
print(f'All-3-high attrition: {all_high_rate:.1%}')
print(f'Relative risk:        {all_low_rate / all_high_rate:.1f}x')

---
## Insight 6 — The Manager Loyalty Anchor

**Hypothesis:** Higher manager tenure ratio (time with current manager / total company tenure)
correlates strongly with *staying*. Good managers are the most underrated retention tool.

In [ ]:
df['MgrBand'] = pd.cut(df['ManagerTenureRatio'],
                        bins=[0, 0.25, 0.50, 0.75, 1.01],
                        labels=['Low (0–25%)', 'Medium (25–50%)', 'High (50–75%)', 'Very High (75%+)'],
                        right=False)

mgr_rates = df.groupby('MgrBand', observed=True)['Attrition'].agg(['mean', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
colors = [PALETTE['left'], PALETTE['neutral'], PALETTE['stayed'], '#1a5276']
bars = ax.bar(mgr_rates['MgrBand'], mgr_rates['mean'] * 100, color=colors, edgecolor='white')
ax.axhline(df['Attrition'].mean() * 100, color='gray', linestyle='--', label='Company avg')
for bar, row in zip(bars, mgr_rates.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{row.mean:.1%}\n(n={row.count})', ha='center', va='bottom', fontsize=9.5)
ax.set_title('INSIGHT 6 — The Manager Loyalty Anchor\n'
             'Employees with stable, long-term managers leave far less',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Attrition Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

low_mgr  = df[df['MgrBand'] == 'Low (0–25%)']['Attrition'].mean()
high_mgr = df[df['MgrBand'] == 'Very High (75%+)']['Attrition'].mean()
print(f'Low manager tenure ratio:  {low_mgr:.1%}')
print(f'High manager tenure ratio: {high_mgr:.1%}')
print(f'Protection factor:         {low_mgr/high_mgr:.1f}x higher departure risk with unstable managers')

---
## Insight 7 — The Stock Option Paradox

**Hypothesis:** Stock option level 1 has the *lowest* attrition — but level 3 (maximum)
does NOT protect retention as expected. "Golden handcuffs" at level 3 breed resentment.

In [ ]:
stock_rates = df.groupby('StockOptionLevel')['Attrition'].agg(['mean', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
colors = [PALETTE['neutral'], PALETTE['stayed'], PALETTE['neutral'], PALETTE['left']]
bars = ax.bar(stock_rates['StockOptionLevel'], stock_rates['mean'] * 100,
              color=colors, edgecolor='white', width=0.5)
for bar, row in zip(bars, stock_rates.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{row.mean:.1%}\n(n={row.count})', ha='center', va='bottom', fontweight='bold')
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['Level 0\n(None)', 'Level 1\n(Low)', 'Level 2\n(Medium)', 'Level 3\n(Max)'])
ax.axhline(df['Attrition'].mean() * 100, color='gray', linestyle='--', label='Company avg')
ax.set_title('INSIGHT 7 — The Stock Option Paradox\n'
             'Level 1 retains best; Level 3 surprisingly does not',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Attrition Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

print('Stock option attrition rates:')
for _, row in stock_rates.iterrows():
    print(f'  Level {int(row["StockOptionLevel"])}: {row["mean"]:.1%}  (n={int(row["count"])})')

---
## Insight 8 — The Distance Gradient

**Hypothesis:** Attrition climbs non-linearly beyond 10 miles from home.
The jump between "Moderate" (6–10 mi) and "Far" (11–20 mi) is the steepest.

> "The commute doesn't just cost time — it costs loyalty."

In [ ]:
dist_smooth = df.groupby('DistanceFromHome')['Attrition'].agg(['mean', 'count']).reset_index()
dist_smooth = dist_smooth[dist_smooth['count'] >= 10]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(dist_smooth['DistanceFromHome'], dist_smooth['mean'] * 100,
                s=dist_smooth['count'] * 2, alpha=0.6, color=PALETTE['left'], edgecolors='white')
# Trend line
z = np.polyfit(dist_smooth['DistanceFromHome'], dist_smooth['mean'] * 100, 2)
p_func = np.poly1d(z)
xs = np.linspace(1, dist_smooth['DistanceFromHome'].max(), 100)
axes[0].plot(xs, p_func(xs), color='darkred', linewidth=2, linestyle='--', label='Quadratic trend')
axes[0].set_title('Attrition Rate vs. Distance from Home', fontweight='bold')
axes[0].set_xlabel('Distance (miles)')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].legend()

tier_rates = df.groupby('DistanceTier', observed=True)['Attrition'].agg(['mean', 'count']).reset_index()
colors = [PALETTE['stayed'], PALETTE['neutral'], PALETTE['left'], '#922b21']
bars = axes[1].bar(tier_rates['DistanceTier'], tier_rates['mean'] * 100,
                   color=colors, edgecolor='white')
for bar, row in zip(bars, tier_rates.itertuples()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{row.mean:.1%}', ha='center', va='bottom', fontweight='bold')
axes[1].axhline(df['Attrition'].mean() * 100, color='gray', linestyle='--', label='Company avg')
axes[1].set_title('Attrition by Distance Tier', fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].legend()

plt.suptitle('INSIGHT 8 — The Distance Gradient', fontsize=14, fontweight='bold', color=PALETTE['left'])
plt.tight_layout()
plt.show()

---
## Insight 9 — The Single Employee Vulnerability

In [ ]:
marital_rates = df.groupby('MaritalStatus', observed=True)['Attrition'].agg(['mean', 'count']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Single': PALETTE['left'], 'Married': PALETTE['stayed'], 'Divorced': PALETTE['neutral']}
bars = axes[0].bar(marital_rates['MaritalStatus'],
                   marital_rates['mean'] * 100,
                   color=[colors[m] for m in marital_rates['MaritalStatus']],
                   edgecolor='white', width=0.5)
axes[0].axhline(df['Attrition'].mean() * 100, color='gray', linestyle='--', label='Company avg')
for bar, row in zip(bars, marital_rates.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{row.mean:.1%}\n(n={row.count})', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Attrition by Marital Status', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].legend()

# Interaction: Marital x Overtime
interact = df.groupby(['MaritalStatus', 'OverTime'], observed=True)['Attrition'].mean().unstack('OverTime')
interact.columns = ['No OT', 'OT']
sns.heatmap(interact * 100, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=axes[1], linewidths=0.5, cbar_kws={'label': 'Attrition %'})
axes[1].set_title('Marital Status × Overtime Interaction', fontweight='bold')
axes[1].set_xlabel('Overtime')
axes[1].set_ylabel('Marital Status')

plt.suptitle('INSIGHT 9 — The Single Employee Vulnerability', fontsize=14, fontweight='bold', color=PALETTE['left'])
plt.tight_layout()
plt.show()

single_ot = df[(df['MaritalStatus']=='Single') & (df['OverTime']==1)]['Attrition'].mean()
print(f'Peak combination — Single + Overtime: {single_ot:.1%} attrition rate')

---
## Insight 10 — Business Travel Burnout

In [ ]:
travel_rates = df.groupby('BusinessTravel', observed=True)['Attrition'].agg(['mean', 'count']).reset_index()
order = ['Non-Travel', 'Travel_Rarely', 'Travel_Frequently']
travel_rates = travel_rates.set_index('BusinessTravel').loc[order].reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = [PALETTE['stayed'], PALETTE['neutral'], PALETTE['left']]
bars = axes[0].bar(travel_rates['BusinessTravel'], travel_rates['mean'] * 100,
                   color=colors, edgecolor='white', width=0.5)
axes[0].axhline(df['Attrition'].mean() * 100, color='gray', linestyle='--', label='Company avg')
for bar, row in zip(bars, travel_rates.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{row.mean:.1%}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Attrition by Business Travel Frequency', fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].legend()

# Travel x Work-Life Balance
travel_wlb = df.groupby(['BusinessTravel', 'WorkLifeBalance'], observed=True)['Attrition'].mean().unstack('BusinessTravel')
travel_wlb = travel_wlb[order]
sns.heatmap(travel_wlb * 100, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=axes[1], linewidths=0.5, cbar_kws={'label': 'Attrition %'})
axes[1].set_title('Attrition: Work-Life Balance × Business Travel', fontweight='bold')

plt.suptitle('INSIGHT 10 — Business Travel Burnout', fontsize=14, fontweight='bold', color=PALETTE['left'])
plt.tight_layout()
plt.show()

freq_bad_wlb = df[(df['BusinessTravel']=='Travel_Frequently') & (df['WorkLifeBalance']==1)]['Attrition'].mean()
print(f'Travel Frequently + Bad Work-Life Balance: {freq_bad_wlb:.1%} attrition')

---
## Executive Summary Dashboard

In [ ]:
risk = pd.read_parquet(ROOT / 'data' / 'gold' / 'risk_scores.parquet')

fig = plt.figure(figsize=(18, 10), facecolor='#1a1a2e')
fig.suptitle('EMPLOYEE ATTRITION — EXECUTIVE INTELLIGENCE REPORT',
             fontsize=16, fontweight='bold', color='white', y=0.97)

KPIs = [
    ('Total Employees', f'{len(df):,}', '#3498db'),
    ('Attrition Rate', f'{df["Attrition"].mean():.1%}', '#e74c3c'),
    ('OT Attrition Rate', f'{df[df["OverTime"]==1]["Attrition"].mean():.1%}', '#e67e22'),
    ('High/Critical Risk', f'{int((risk["RiskBand"].isin(["High", "Critical"])).sum()):,}', '#8e44ad'),
    ('Sales Rep Rate', f'{df[df["JobRole"]=="Sales Representative"]["Attrition"].mean():.1%}', '#e74c3c'),
    ('All-Low Sat. Rate', f'{df[df["AllLow"]==1]["Attrition"].mean():.1%}', '#922b21'),
]

for i, (label, value, color) in enumerate(KPIs):
    ax = fig.add_axes([0.02 + (i % 3) * 0.33, 0.75 - (i // 3) * 0.18, 0.28, 0.15])
    ax.set_facecolor(color)
    ax.text(0.5, 0.65, value, transform=ax.transAxes, ha='center', va='center',
            fontsize=22, fontweight='bold', color='white')
    ax.text(0.5, 0.2, label, transform=ax.transAxes, ha='center', va='center',
            fontsize=10, color='white', alpha=0.9)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)

# Risk band bar
ax_risk = fig.add_axes([0.02, 0.08, 0.45, 0.35])
ax_risk.set_facecolor('#16213e')
band_counts = risk['RiskBand'].value_counts().reindex(['Low', 'Moderate', 'High', 'Critical'])
bars = ax_risk.bar(band_counts.index, band_counts.values,
                   color=['#27ae60', '#f39c12', '#e67e22', '#e74c3c'], edgecolor='#1a1a2e', linewidth=1.5)
for bar, val in zip(bars, band_counts.values):
    ax_risk.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(val), ha='center', color='white', fontweight='bold')
ax_risk.set_facecolor('#16213e')
ax_risk.tick_params(colors='white')
ax_risk.set_title('Flight Risk Band Distribution', color='white', fontweight='bold')
ax_risk.set_ylabel('Count', color='white')
for spine in ax_risk.spines.values(): spine.set_color('#16213e')

# Top 5 risky roles
ax_roles = fig.add_axes([0.52, 0.08, 0.46, 0.35])
ax_roles.set_facecolor('#16213e')
top_roles = df.groupby('JobRole', observed=True)['Attrition'].mean().nlargest(5)
bars = ax_roles.barh(top_roles.index, top_roles.values * 100,
                     color='#e74c3c', edgecolor='#1a1a2e')
for bar, val in zip(bars, top_roles.values):
    ax_roles.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                  f'{val:.1%}', va='center', color='white', fontsize=9)
ax_roles.set_facecolor('#16213e')
ax_roles.tick_params(colors='white')
ax_roles.set_title('Top 5 High-Attrition Roles', color='white', fontweight='bold')
ax_roles.set_xlabel('Attrition Rate (%)', color='white')
for spine in ax_roles.spines.values(): spine.set_color('#16213e')

plt.savefig(ROOT / 'docs' / 'executive_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('Executive dashboard saved to docs/executive_dashboard.png')

---
## Action Recommendations

| Priority | Insight | Recommended Action |
|---|---|---|
| P0 | Overtime Trap | Audit all overtime-heavy roles quarterly. Cap OT at 15% of role headcount. |
| P0 | Satisfaction Trinity | Flag employees scoring ≤2 in all 3 dimensions for immediate manager 1:1. |
| P1 | Salary Compression | Review pay equity by job level annually. Correct below-median outliers. |
| P1 | Promotion Stagnation | Enforce max 3-year promotion review cycle for all IC roles. |
| P1 | 3-Year Crisis | Structured check-ins at months 3, 6, 18, and 36 for all new hires. |
| P2 | Manager Loyalty | Include manager stability in HR risk scoring. Frequent mgr churn = red flag. |
| P2 | Business Travel | Cap Travel_Frequently to 20% of team composition. Rotate travel duty. |
| P2 | Single Employees | Targeted ERG and social programs for single employees in high-OT roles. |
| P3 | Distance Gradient | Consider remote/hybrid options for employees > 15 miles from office. |
| P3 | Stock Options | Revisit Level 3 vesting schedule — perceived ceiling may demotivate. |